# **Preprocessing Experiments Notebook**

This notebook is to test the web scraping frameworks for extracting posts from X, Facebook, Reddit, and Tiktok in the Philippines for possible symptoms of pulmonary dieseases (COVID-19, TB, Pneuomonia)

# Data Collection Overview

---

## 🌐 Languages & Dialects

| # | Language / Dialect |
|---|-------------------|
| 1 | English |
| 2 | Tagalog |
| 3 | Cebuano |
| 4 | Hiligaynon |
| 5 | Ilocano |

---

## 🔑 Keywords

> Refer to the **`/data`** folder for the complete keyword list.

---

## 📡 Data Sources

| # | Platform | Status |
|---|----------|--------|
| 1 | X / Twitter | Existing |
| 2 | Facebook | Existing |
| 3 | Reddit | Existing |
| 4 | TikTok | Existing |
| 5 | Threads | 🆕 New |

In [172]:
# Dependencies for Data
import uuid
import html
import re
import numpy as np
import pandas as pd
from pathlib import Path


def normalize_columns(df):
    """
    Normalize column names by removing BOMs, quotes, and surrounding whitespace.
    """
    df = df.copy()
    df.columns = [str(c).lstrip("\ufeff").strip().strip('"') for c in df.columns]
    return df


def coalesce_columns(df, candidates):
    """Return one series by coalescing candidate columns row by row."""
    available = [c for c in candidates if c in df.columns]
    if not available:
        return pd.Series(pd.NA, index=df.index, dtype="object")

    out = pd.Series(pd.NA, index=df.index, dtype="object")
    for col in available:
        out = out.fillna(df[col])
    return out


def to_nullable_int(series):
    """Convert a series to pandas nullable integer dtype (Int64)."""
    return pd.to_numeric(series, errors="coerce").astype("Int64")


In [173]:
# Dependecies for Language Detection
import fasttext as ft

In [174]:
# posts table schema template from docs/data schema.md
POSTS_COLUMNS = [
    "id",
    "source",
    "external_post_id",
    "text",
    "cleaned_text",
    "language",
    "date_posted",
    "date_collected",
    "view_count",
    "like_count",
    "share_count",
    "comment_count",
    "created_at",
    "updated_at",
]

posts_df = pd.DataFrame(columns=POSTS_COLUMNS)
posts_df


,id,source,external_post_id,text,cleaned_text,language,date_posted,date_collected,view_count,like_count,share_count,comment_count,created_at,updated_at


## Importing Dataset

### X/Twitter

In [175]:

data_dir = Path('../data/raw/twitter')
csv_files = sorted(data_dir.glob('*.csv'))

dfs = [normalize_columns(pd.read_csv(f)) for f in csv_files]
twitter_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

required_cols = ["created_at", "id", "text"]
missing_twitter = [c for c in required_cols if c not in twitter_df.columns]
if missing_twitter:
    raise KeyError(f"Missing Twitter columns: {missing_twitter}. Found: {twitter_df.columns.tolist()}")

# add column for source website
twitter_df["source"] = "twitter"
twitter_df["id"] = twitter_df["id"].astype(str).str.replace("tweet-", "", regex=False).str.strip()

# Extract engagement metrics with fallback aliases
twitter_df["view_count"] = to_nullable_int(coalesce_columns(twitter_df, ["view_count", "views_count", "impression_count"]))
twitter_df["like_count"] = to_nullable_int(coalesce_columns(twitter_df, ["like_count", "favorite_count", "favorites_count", "heart_count", "hearts_count"]))
twitter_df["comment_count"] = to_nullable_int(coalesce_columns(twitter_df, ["comment_count", "reply_count", "replies_count"]))
twitter_df["share_count"] = to_nullable_int(coalesce_columns(twitter_df, ["share_count", "retweet_count", "retweets_count", "repost_count"]))

twitter_df = twitter_df[["created_at", "id", "text", "source", "view_count", "like_count", "share_count", "comment_count"]]
twitter_df


,created_at,id,text,source,view_count,like_count,share_count,comment_count
0,Sun Sep 21 23:59:43 +0000 2025,1969914468458185043,https://t.co/jbO5TDH1nM\n\n牛乳石鹸コラボユニボールワンP\n\n...,twitter,<NA>,<NA>,<NA>,<NA>
1,Sun Sep 21 23:53:27 +0000 2025,1969912891831939529,anong gusto mo gawin niya makipagbarda sa mga ...,twitter,<NA>,<NA>,<NA>,<NA>
2,Sun Sep 21 23:28:01 +0000 2025,1969906492611645849,grabe na hutoy sa ubo thanks mama cels sa pag ...,twitter,<NA>,<NA>,<NA>,<NA>
3,Sun Sep 21 23:26:11 +0000 2025,1969906030298763648,@ubo_ub @MaseDenver Trautman sucks ass,twitter,<NA>,<NA>,<NA>,<NA>
4,Sun Sep 21 23:19:27 +0000 2025,1969904337368256777,@MaseDenver @MaseDenver thoughts on the offici...,twitter,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...
28928,2025-11-16T02:00:23.000Z,1989876169949561114,"tangina nitong tatay ni Prem. chararat na nga,...",twitter,<NA>,0,<NA>,<NA>
28929,2025-11-15T14:52:59.000Z,1989708213537640695,@073TUMELO @Solomon_mabee @Patriot_S_A Tsokotl...,twitter,<NA>,2,<NA>,<NA>
28930,2025-11-15T12:19:34.000Z,1989669604759932936,"@YolzYako Congratulations to him, uNtate Maput...",twitter,<NA>,0,<NA>,<NA>
28931,2025-11-15T11:18:59.000Z,1989654357584089237,Your son ke John Maputla??? https://t.co/dwt0j...,twitter,<NA>,0,<NA>,<NA>


### Facebook

In [176]:
# Importing the CSVs for Facebook 
facebook_dir = Path('../data/raw/facebook')
facebook_csv_files = sorted(facebook_dir.glob('*.csv'))
facebook_dfs = [normalize_columns(pd.read_csv(f)) for f in facebook_csv_files]
facebook_df = pd.concat(facebook_dfs, ignore_index=True) if facebook_dfs else pd.DataFrame()


In [177]:
# Accept `created_at`, `timestamp`, or `timestamps` for Facebook time values
time_candidates = ["created_at", "timestamp", "timestamps"]
available_time_cols = [c for c in time_candidates if c in facebook_df.columns]
if not available_time_cols:
    raise KeyError(
        f"Missing Facebook time column. Expected one of {time_candidates}. Found: {facebook_df.columns.tolist()}"
    )

required_cols = ["post_id", "message"]
missing = [c for c in required_cols if c not in facebook_df.columns]
if missing:
    raise KeyError(f"Missing Facebook columns: {missing}. Found: {facebook_df.columns.tolist()}")

# Capture engagement metrics before narrowing columns
fb_view_count = to_nullable_int(coalesce_columns(facebook_df, ["view_count", "views_count"]))
fb_like_count = to_nullable_int(coalesce_columns(facebook_df, ["like_count", "likes", "reaction_count", "reactions_count"]))
fb_comment_count = to_nullable_int(coalesce_columns(facebook_df, ["comment_count", "comments_count", "reply_count", "replies_count"]))
fb_share_count = to_nullable_int(coalesce_columns(facebook_df, ["share_count", "shares_count", "repost_count"]))

# Coalesce time columns row-by-row so mixed CSV formats still work
facebook_time_raw = pd.Series(pd.NA, index=facebook_df.index, dtype="object")
for col in available_time_cols:
    facebook_time_raw = facebook_time_raw.fillna(facebook_df[col])

facebook_df = facebook_df[["post_id", "message"]].copy()
facebook_df["created_at"] = facebook_time_raw
facebook_df = facebook_df.rename(columns={"post_id": "id", "message": "text"})

# Parse unix seconds first, then fall back to generic datetime parsing
raw_fb_time = facebook_df["created_at"].astype(str).str.strip()
fb_unix = pd.to_datetime(pd.to_numeric(raw_fb_time, errors="coerce"), unit="s", errors="coerce", utc=True)
fb_fallback = pd.to_datetime(raw_fb_time, errors="coerce", utc=True)
facebook_df["created_at"] = fb_unix.fillna(fb_fallback)

# add column for source website + separate engagement columns
facebook_df["source"] = "facebook"
facebook_df["view_count"] = fb_view_count
facebook_df["like_count"] = fb_like_count
facebook_df["comment_count"] = fb_comment_count
facebook_df["share_count"] = fb_share_count
facebook_df = facebook_df[["created_at", "id", "text", "source", "view_count", "like_count", "share_count", "comment_count"]]


/var/folders/mq/j8mtkq590gl29rkr5th7fzdh0000gp/T/ipykernel_13797/3221092446.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fb_fallback = pd.to_datetime(raw_fb_time, errors="coerce", utc=True)


In [178]:
facebook_df

,created_at,id,text,source,view_count,like_count,share_count,comment_count
0,2026-01-25 04:29:07+00:00,4234805846762213,Mga Dapat Gawin Kapag may Lagnat ang Baby (0–1...,facebook,<NA>,1178,<NA>,<NA>
1,2026-02-05 15:44:13+00:00,1989725038640084,"mga mii, normal langba pa balik balik lagnat n...",facebook,<NA>,19,<NA>,<NA>
2,2026-02-19 08:56:46+00:00,26084263801204435,Wag ipagsawalang bahala ang simpleng ubo at la...,facebook,<NA>,14071,<NA>,<NA>
3,2026-02-15 15:39:10+00:00,2531940147243104,AWARENESS FOR RABIES ‼️\n\nMy kuya was bitten ...,facebook,<NA>,6405,<NA>,<NA>
4,2026-01-15 22:12:46+00:00,856786103906459,"Napakabisang gamot sa ubo, sipon, lagnat o tra...",facebook,<NA>,864,<NA>,<NA>
...,...,...,...,...,...,...,...,...
95,2025-12-01 12:06:46+00:00,1556836565393789,pabalik balik lagnat ko grr ayoko na pls,facebook,<NA>,2,<NA>,<NA>
96,2026-02-14 03:48:03+00:00,2360679151111363,Mga mi mag lagnat ba pag 37.4 to 37.2 Ang temp...,facebook,<NA>,4,<NA>,<NA>
97,2025-12-13 07:58:25+00:00,2293443264501619,Hellow mga mi. Permission to ask everyone and ...,facebook,<NA>,20,<NA>,<NA>
98,2026-02-04 00:17:15+00:00,1988233018789286,hellow po mga mie ano po kayang gamot sa pabal...,facebook,<NA>,6,<NA>,<NA>


### Threads

In [179]:
# Importing the csvs for threads
thread_dir = Path('../data/raw/threads')
thread_csv_files = sorted(thread_dir.glob('*.csv'))
thread_dfs = [normalize_columns(pd.read_csv(f)) for f in thread_csv_files]
threads_df = pd.concat(thread_dfs, ignore_index=True) if thread_dfs else pd.DataFrame()

# Accept `created_at`, `timestamp`, or `timestamps` from Threads exports
thread_time_candidates = ["created_at", "timestamp", "timestamps"]
thread_time_cols = [c for c in thread_time_candidates if c in threads_df.columns]
if not thread_time_cols:
    raise KeyError(
        f"Missing Threads time column. Expected one of {thread_time_candidates}. Found: {threads_df.columns.tolist()}"
    )

thread_required_cols = ["id", "text"]
thread_missing = [c for c in thread_required_cols if c not in threads_df.columns]
if thread_missing:
    raise KeyError(f"Missing Threads columns: {thread_missing}. Found: {threads_df.columns.tolist()}")

# Capture engagement metrics before narrowing columns
thread_view_count = to_nullable_int(coalesce_columns(threads_df, ["view_count", "views_count", "play_count"]))
thread_like_count = to_nullable_int(coalesce_columns(threads_df, ["like_count", "favorite_count", "favorites_count", "heart_count", "hearts_count"]))
thread_comment_count = to_nullable_int(coalesce_columns(threads_df, ["comment_count", "reply_count", "replies_count"]))
thread_share_count = to_nullable_int(coalesce_columns(threads_df, ["share_count", "repost_count", "retweet_count"]))

# Coalesce time columns row-by-row so mixed CSV formats still work
threads_time_raw = pd.Series(pd.NA, index=threads_df.index, dtype="object")
for col in thread_time_cols:
    threads_time_raw = threads_time_raw.fillna(threads_df[col])

threads_df = threads_df[["id", "text"]].copy()
threads_df["created_at"] = threads_time_raw

# Parse unix seconds first, then fall back to generic datetime parsing
raw_thread_time = threads_df["created_at"].astype(str).str.strip()
thread_unix = pd.to_datetime(pd.to_numeric(raw_thread_time, errors="coerce"), unit="s", errors="coerce", utc=True)
thread_fallback = pd.to_datetime(raw_thread_time, errors="coerce", utc=True)
threads_df["created_at"] = thread_unix.fillna(thread_fallback)

# add column for source website + separate engagement columns
threads_df["source"] = "threads"
threads_df["view_count"] = thread_view_count
threads_df["like_count"] = thread_like_count
threads_df["comment_count"] = thread_comment_count
threads_df["share_count"] = thread_share_count
threads_df = threads_df[["created_at", "id", "text", "source", "view_count", "like_count", "share_count", "comment_count"]]


/var/folders/mq/j8mtkq590gl29rkr5th7fzdh0000gp/T/ipykernel_13797/2065193187.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  thread_fallback = pd.to_datetime(raw_thread_time, errors="coerce", utc=True)


In [180]:
threads_df

,created_at,id,text,source,view_count,like_count,share_count,comment_count
0,2026-02-23 00:03:39+00:00,3838501173108437570,I Beg of you to change the name of Usher syndr...,threads,<NA>,<NA>,<NA>,<NA>
1,2026-02-23 00:03:16+00:00,3838500978249470064,I Beg of you to change the name of Usher syndr...,threads,<NA>,<NA>,<NA>,<NA>
2,2026-02-22 16:31:26+00:00,3838273568287930837,Congress’ fate in 2027 was already on display ...,threads,<NA>,<NA>,<NA>,<NA>
3,2026-02-22 18:45:39+00:00,3838340719379805400,Identify the person who kick out corona virus,threads,<NA>,<NA>,<NA>,<NA>
4,2026-02-22 17:16:51+00:00,3838296390962784666,"Aqui reportando,todo encuanto,a los virus que ...",threads,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...
4982,2026-03-23 01:56:17+00:00,88bc9893cbc99cf0220e53644f4865275cd1bdf6,literally_just_a_jet\n4h\nHaii!\nBmf/bmgf\n15\...,threads,<NA>,<NA>,<NA>,<NA>
4983,2026-03-22 09:08:19+00:00,58434d1c236f8d85ac9050f6520d669e7d99fece,daily_super_recipes\n21h\n🍗 Rosemary Butter Ch...,threads,<NA>,<NA>,<NA>,<NA>
4984,2026-03-23 03:01:36+00:00,7e4fa9fff7f2970e96f1f02816d6b8ee3dd607d8,rachelle_after_dark___\n3h\nNever dated white ...,threads,<NA>,<NA>,<NA>,<NA>
4985,2026-03-22 20:30:27+00:00,5a71633cf2b41e1dedafd6458727703d99ed91e2,easwesan\nHudson Williams\n9h\nall u bitches (...,threads,<NA>,<NA>,<NA>,<NA>


### Reddit

In [181]:
# Importing the CSVs for Reddit
reddit_dir = Path('../data/raw/reddit')
reddit_csv_files = sorted(reddit_dir.glob('*.csv'))
reddit_dfs = [normalize_columns(pd.read_csv(f)) for f in reddit_csv_files]
reddit_df = pd.concat(reddit_dfs, ignore_index=True) if reddit_dfs else pd.DataFrame()

reddit_required_cols = ["title", "created"]
reddit_missing = [c for c in reddit_required_cols if c not in reddit_df.columns]
if reddit_missing:
    raise KeyError(f"Missing Reddit columns: {reddit_missing}. Found: {reddit_df.columns.tolist()}")

# Build text from title + selftext/body if available
reddit_title = coalesce_columns(reddit_df, ["title"]).fillna("").astype(str).str.strip()
reddit_body = coalesce_columns(reddit_df, ["selftext", "text", "body"]).fillna("").astype(str).str.strip()
reddit_text = (reddit_title + "\n" + reddit_body).str.strip()
reddit_text = reddit_text.str.replace(r"\n+", "\n", regex=True)

# Prefer existing id columns, then parse from URL, then fall back to deterministic row ids
reddit_ids = coalesce_columns(reddit_df, ["id", "post_id", "name"]).astype("object")
reddit_ids = reddit_ids.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
if "url" in reddit_df.columns:
    url_ids = reddit_df["url"].astype(str).str.extract(r"/comments/([^/]+)/", expand=False)
    reddit_ids = reddit_ids.fillna(url_ids)
fallback_ids = pd.Series([f"reddit-{i}" for i in range(len(reddit_df))], index=reddit_df.index, dtype="object")
reddit_ids = reddit_ids.fillna(fallback_ids).astype(str).str.strip()

raw_reddit_time = coalesce_columns(reddit_df, ["created_at", "created", "timestamp"]).astype(str).str.strip()
reddit_unix = pd.to_datetime(pd.to_numeric(raw_reddit_time, errors="coerce"), unit="s", errors="coerce", utc=True)
reddit_fallback = pd.to_datetime(raw_reddit_time, errors="coerce", utc=True)
reddit_created_at = reddit_unix.fillna(reddit_fallback)

reddit_view_count = to_nullable_int(coalesce_columns(reddit_df, ["view_count", "views_count"]))
reddit_like_count = to_nullable_int(coalesce_columns(reddit_df, ["like_count", "likes", "ups", "upvotes", "score"]))
reddit_comment_count = to_nullable_int(coalesce_columns(reddit_df, ["comment_count", "num_comments", "comments_count"]))
reddit_share_count = to_nullable_int(coalesce_columns(reddit_df, ["share_count", "shares_count"]))

reddit_df = pd.DataFrame({
    "created_at": reddit_created_at,
    "id": reddit_ids,
    "text": reddit_text,
    "source": "reddit",
    "view_count": reddit_view_count,
    "like_count": reddit_like_count,
    "comment_count": reddit_comment_count,
    "share_count": reddit_share_count,
})
reddit_df

,created_at,id,text,source,view_count,like_count,comment_count,share_count
0,2025-10-24 01:10:07+00:00,1oekmr6,"Lagnat, how to remedy it at home?\n. Ano remed...",reddit,<NA>,1,<NA>,<NA>
1,2025-10-16 04:58:21+00:00,1o7xecc,Lagnat 40.3\nHi mga mommies! My 15 mon old is ...,reddit,<NA>,1,<NA>,<NA>
2,2024-12-14 21:02:47+00:00,1hecfqq,Pano alagaan ang may lagnat?\nMay lagnat GF ko...,reddit,<NA>,16,<NA>,<NA>
3,2025-07-30 11:28:06+00:00,1md423j,Pabalik-balik na lagnat\nMagandang araw po! Co...,reddit,<NA>,1,<NA>,<NA>
4,2025-10-03 08:45:09+00:00,1nwtqda,Saw my bf's chatgpt history and it made me cry...,reddit,<NA>,6424,<NA>,<NA>
...,...,...,...,...,...,...,...,...
24897,2026-04-03 06:20:59+00:00,1savob7,Siento que no encajo\nEscribo esto para más qu...,reddit,<NA>,2,<NA>,<NA>
24898,2026-04-02 14:44:52+00:00,1sa9yoo,"La mayoría de los ""trucos migratorios"" son una...",reddit,<NA>,1,<NA>,<NA>
24899,2026-04-02 08:37:19+00:00,1sa2klf,"Quebrarse,esperar o aceptar\nCreo que no es un...",reddit,<NA>,11,<NA>,<NA>
24900,2026-04-02 07:36:00+00:00,1sa168t,Es correcto este ultimo mensaje?\nNo quiero su...,reddit,<NA>,0,<NA>,<NA>


### TikTok

In [182]:
# Importing the CSVs for TikTok
tiktok_dir = Path('../data/raw/tiktok')
tiktok_csv_files = sorted(tiktok_dir.glob('*.csv'))
tiktok_dfs = [normalize_columns(pd.read_csv(f)) for f in tiktok_csv_files]
tiktok_df = pd.concat(tiktok_dfs, ignore_index=True) if tiktok_dfs else pd.DataFrame()

tiktok_required_cols = ["id", "text"]
tiktok_missing = [c for c in tiktok_required_cols if c not in tiktok_df.columns]
if tiktok_missing:
    raise KeyError(f"Missing TikTok columns: {tiktok_missing}. Found: {tiktok_df.columns.tolist()}")

raw_tiktok_time = coalesce_columns(
    tiktok_df,
    ["createTimeISO", "create_time_iso", "created_at", "createTime", "create_time", "timestamp"],
).astype(str).str.strip()
tiktok_unix = pd.to_datetime(pd.to_numeric(raw_tiktok_time, errors="coerce"), unit="s", errors="coerce", utc=True)
tiktok_fallback = pd.to_datetime(raw_tiktok_time, errors="coerce", utc=True)
tiktok_created_at = tiktok_unix.fillna(tiktok_fallback)

tiktok_ids = coalesce_columns(tiktok_df, ["id", "aweme_id", "post_id"]).astype("object")
tiktok_ids = tiktok_ids.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
tiktok_fallback_ids = pd.Series([f"tiktok-{i}" for i in range(len(tiktok_df))], index=tiktok_df.index, dtype="object")
tiktok_ids = tiktok_ids.fillna(tiktok_fallback_ids).astype(str).str.strip()

tiktok_text = coalesce_columns(tiktok_df, ["text", "desc", "description"]).fillna("").astype(str).str.strip()
tiktok_view_count = to_nullable_int(coalesce_columns(tiktok_df, ["view_count", "views_count", "playCount", "play_count"]))
tiktok_like_count = to_nullable_int(coalesce_columns(tiktok_df, ["like_count", "likes_count", "diggCount", "digg_count", "favorite_count", "favorites_count"]))
tiktok_comment_count = to_nullable_int(coalesce_columns(tiktok_df, ["comment_count", "comments_count", "commentCount"]))
tiktok_share_count = to_nullable_int(coalesce_columns(tiktok_df, ["share_count", "shares_count", "shareCount", "repostCount", "repost_count"]))

tiktok_df = pd.DataFrame({
    "created_at": tiktok_created_at,
    "id": tiktok_ids,
    "text": tiktok_text,
    "source": "tiktok",
    "view_count": tiktok_view_count,
    "like_count": tiktok_like_count,
    "comment_count": tiktok_comment_count,
    "share_count": tiktok_share_count,
})
tiktok_df



,created_at,id,text,source,view_count,like_count,comment_count,share_count
0,2025-11-08 11:54:18+00:00,7570321304935583007,The greatest #slap of all time 🤯 The #PowerSla...,tiktok,567300,<NA>,157,2404
1,2025-12-11 04:43:13+00:00,7582456311653092629,"KO 😱🔥 Videos completos en YouTube: ""El Nea Ofi...",tiktok,4400000,<NA>,508,10200
2,2025-11-07 00:44:35+00:00,7569777888577195286,#onthisday KO’d #ko #knockout #knockouts #wome...,tiktok,70000,<NA>,17,11
3,2026-02-18 10:31:33+00:00,7608150958882835734,Moments before deenthegreat got KO'd a second ...,tiktok,1800000,<NA>,167,269
4,2025-09-08 10:46:21+00:00,7547667900442610966,Kid KOed 😬😬#ko #knockout #fyp #muaythai,tiktok,82200,<NA>,26,120
...,...,...,...,...,...,...,...,...
805,2026-03-09 07:09:26+00:00,7615149486167330069,Parang ice tubig sa lamig 😆❄️ pero yung view i...,tiktok,2926,<NA>,6,2
806,2026-03-08 09:08:20+00:00,7614809016413211924,#duet with @Cortez motovlog😭😭😭😭😭😭😭😭😭😭😭😭😭,tiktok,4885,<NA>,8,16
807,2026-03-09 14:34:13+00:00,7615264105179499796,#CapCut,tiktok,100,<NA>,4,1
808,2026-03-08 06:37:29+00:00,7614770168417291541,#CapCut #fyp mga mananabtan 🙏🏼 #fypppppppppppp...,tiktok,387,<NA>,4,3


# Merging Datasets

In [183]:
# Add the source dataframes into a list for easier processing
scraped_dfs = [twitter_df, threads_df, facebook_df, reddit_df, tiktok_df]

# Keep IDs as strings to support numeric and non-numeric platform ids
for df in scraped_dfs:
    df["id"] = df["id"].astype(str).str.strip()

# Merging the dataframes into one
merged_df = pd.concat(scraped_dfs, ignore_index=True)


merged_df.to_csv('../data/combined/combined_data.csv', index=False)



In [184]:
merged_df.head()

,created_at,id,text,source,view_count,like_count,share_count,comment_count
0,Sun Sep 21 23:59:43 +0000 2025,1969914468458185043,https://t.co/jbO5TDH1nM\n\n牛乳石鹸コラボユニボールワンP\n\n...,twitter,<NA>,<NA>,<NA>,<NA>
1,Sun Sep 21 23:53:27 +0000 2025,1969912891831939529,anong gusto mo gawin niya makipagbarda sa mga ...,twitter,<NA>,<NA>,<NA>,<NA>
2,Sun Sep 21 23:28:01 +0000 2025,1969906492611645849,grabe na hutoy sa ubo thanks mama cels sa pag ...,twitter,<NA>,<NA>,<NA>,<NA>
3,Sun Sep 21 23:26:11 +0000 2025,1969906030298763648,@ubo_ub @MaseDenver Trautman sucks ass,twitter,<NA>,<NA>,<NA>,<NA>
4,Sun Sep 21 23:19:27 +0000 2025,1969904337368256777,@MaseDenver @MaseDenver thoughts on the offici...,twitter,<NA>,<NA>,<NA>,<NA>


In [185]:
merged_df_cleaned = merged_df.copy()

In [186]:
def drop_invalid_raw_text(df):
    """Drop rows with missing or blank scraped text before text normalization."""
    df = df.copy()
    start_rows = len(df)
    text = df["text"]
    valid_text = text.notna() & text.astype(str).str.strip().ne("")
    df = df.loc[valid_text].copy()

    return df, {
        "starting_rows": start_rows,
        "rows_dropped_missing_text": int((~valid_text).sum()),
    }


def deduplicate_posts(df):
    """Deduplicate by platform and external id so ids from different sources do not collide."""
    df = df.copy()
    df["source"] = df["source"].astype(str).str.strip().str.lower()
    df["id"] = df["id"].astype(str).str.strip()
    duplicate_post_ids = df.duplicated(subset=["source", "id"], keep="first")

    return df.loc[~duplicate_post_ids].copy(), {
        "duplicate_source_id_rows": int(duplicate_post_ids.sum()),
    }


merged_df_cleaned, raw_text_summary = drop_invalid_raw_text(merged_df_cleaned)
merged_df_cleaned, source_id_summary = deduplicate_posts(merged_df_cleaned)


In [187]:
def normalize_scraped_text(value):
    """Normalize scraped text while preserving the annotation cleaning contract."""
    if pd.isna(value):
        return ""

    text = html.unescape(html.unescape(str(value)))
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"[\u200b-\u200f\ufeff]", "", text)
    text = re.sub(r"[\x00-\x1f\x7f]", " ", text)
    text = re.sub(r"^\s*rt\s+@\w+\s*:\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"!\[[^\]]*\]\([^)]*\)", " ", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]*$", r"\1", text)
    text = re.sub(r"https?://\S+|//www\.\S+|www\.\S+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^\x00-\x7F]+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"https?://\S+|//www\.\S+|www\.\S+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"(^|\s)#+(?=\s|$)", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


def add_cleaned_text(df):
    """Create cleaned_text from raw scraped text."""
    df = df.copy()
    df["cleaned_text"] = df["text"].map(normalize_scraped_text)
    return df


def drop_invalid_cleaned_text(df):
    """Drop cleaned rows that are empty, low-information, or exact duplicates."""
    df = df.copy()
    cleaned = df["cleaned_text"].fillna("").astype(str).str.strip()
    has_ascii_letter = cleaned.str.contains(r"[a-zA-Z]", regex=True)
    valid_cleaned_text = cleaned.ne("") & has_ascii_letter
    invalid_cleaned_rows = ~valid_cleaned_text

    df = df.loc[valid_cleaned_text].copy()
    duplicate_cleaned_text = df.duplicated(subset=["cleaned_text"], keep="first")
    df = df.loc[~duplicate_cleaned_text].copy()

    return df, {
        "rows_dropped_empty_or_invalid_cleaned_text": int(invalid_cleaned_rows.sum()),
        "duplicate_cleaned_text_rows": int(duplicate_cleaned_text.sum()),
    }


merged_df_cleaned = add_cleaned_text(merged_df_cleaned)
merged_df_cleaned, cleaned_text_summary = drop_invalid_cleaned_text(merged_df_cleaned)

preprocessing_summary = {
    **raw_text_summary,
    **source_id_summary,
    **cleaned_text_summary,
    "final_rows": len(merged_df_cleaned),
}


In [188]:
def normalize_df(df):
    """
    Normalize merged raw data before mapping to the posts table schema.
    Supports Twitter date strings and unix timestamps from Threads/Facebook.
    """
    df = df.copy()
    df["id"] = df["id"].astype(str).str.strip()

    raw = df["created_at"].astype(str).str.strip()
    twitter_like = pd.to_datetime(
        raw.str.replace(r" \+\d{4}$", "", regex=True),
        format="%a %b %d %H:%M:%S %Y",
        errors="coerce",
        utc=True,
    )
    unix_like = pd.to_datetime(
        pd.to_numeric(raw, errors="coerce"),
        unit="s",
        errors="coerce",
        utc=True,
    )
    fallback = pd.to_datetime(raw, errors="coerce", utc=True)

    parsed_created_at = twitter_like.fillna(unix_like).fillna(fallback)
    df["created_at"] = parsed_created_at
    df["date_posted"] = parsed_created_at
    df["text"] = df["text"].astype(str).str.replace(r"\n", " ", regex=True).str.strip()

    if "cleaned_text" not in df.columns:
        df["cleaned_text"] = df["text"]
    df["cleaned_text"] = df["cleaned_text"].astype(str).str.replace(r"\n", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()

    df["external_post_id"] = df["id"]

    for metric in ["view_count", "like_count", "share_count", "comment_count"]:
        if metric not in df.columns:
            df[metric] = pd.NA
        df[metric] = to_nullable_int(df[metric])

    df = df[[
        "source",
        "external_post_id",
        "text",
        "cleaned_text",
        "date_posted",
        "created_at",
        "view_count",
        "like_count",
        "share_count",
        "comment_count",
    ]]
    return df

merged_df_cleaned = normalize_df(merged_df_cleaned)


/var/folders/mq/j8mtkq590gl29rkr5th7fzdh0000gp/T/ipykernel_13797/2222233791.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fallback = pd.to_datetime(raw, errors="coerce", utc=True)


In [189]:
preprocessing_summary_df = pd.DataFrame(
    list(preprocessing_summary.items()),
    columns=["metric", "value"],
)

source_counts_df = (
    merged_df_cleaned["source"]
    .value_counts(dropna=False)
    .rename_axis("source")
    .reset_index(name="final_rows")
)

display(preprocessing_summary_df)
display(source_counts_df)
merged_df_cleaned.head()


,metric,value
0,starting_rows,59732
1,rows_dropped_missing_text,89
2,duplicate_source_id_rows,2253
3,rows_dropped_empty_or_invalid_cleaned_text,1015
4,duplicate_cleaned_text_rows,10081
5,final_rows,46294


,source,final_rows
0,reddit,23740
1,twitter,18070
2,threads,3798
3,tiktok,609
4,facebook,77


,source,external_post_id,text,cleaned_text,date_posted,created_at,view_count,like_count,share_count,comment_count
0,twitter,1969914468458185043,https://t.co/jbO5TDH1nM 牛乳石鹸コラボユニボールワンP 1家族1...,p 11,2025-09-21 23:59:43+00:00,2025-09-21 23:59:43+00:00,<NA>,<NA>,<NA>,<NA>
1,twitter,1969912891831939529,anong gusto mo gawin niya makipagbarda sa mga ...,anong gusto mo gawin niya makipagbarda sa mga ...,2025-09-21 23:53:27+00:00,2025-09-21 23:53:27+00:00,<NA>,<NA>,<NA>,<NA>
2,twitter,1969906492611645849,grabe na hutoy sa ubo thanks mama cels sa pag ...,grabe na hutoy sa ubo thanks mama cels sa pag ...,2025-09-21 23:28:01+00:00,2025-09-21 23:28:01+00:00,<NA>,<NA>,<NA>,<NA>
3,twitter,1969906030298763648,@ubo_ub @MaseDenver Trautman sucks ass,trautman sucks ass,2025-09-21 23:26:11+00:00,2025-09-21 23:26:11+00:00,<NA>,<NA>,<NA>,<NA>
4,twitter,1969904337368256777,@MaseDenver @MaseDenver thoughts on the offici...,thoughts on the officiating crew tonight? that...,2025-09-21 23:19:27+00:00,2025-09-21 23:19:27+00:00,<NA>,<NA>,<NA>,<NA>


## Language Detection

In [190]:

# Load the pre-trained language identification model
model_candidates = [
    Path('models/lid.176.bin'),
    Path('models/lid.176.ftz'),
    Path('../models/lid.176.bin'),
    Path('../models/lid.176.ftz'),
]
model_path = next((path for path in model_candidates if path.exists()), None)
if model_path is None:
    raise FileNotFoundError(
        'Could not find a FastText language model. Checked: '
        + ', '.join(str(path) for path in model_candidates)
    )

model = ft.load_model(str(model_path))
language_predictions = model.predict(merged_df_cleaned['cleaned_text'].tolist(), k=1)


In [191]:
# Extract the predicted language labels and confidence scores
predicted_languages = [label[0].replace('__label__', '') for label in language_predictions[0]]
confidence_scores = language_predictions[1]

# Add the predicted languages and confidence scores to the DataFrame
merged_df_cleaned['language'] = predicted_languages

In [192]:
language_codes = ['en','tl', 'ceb', 'fil', 'ilo', 'hil']

# Filter the DataFrame to keep only rows where the predicted language is English, Tagalog, Cebuano, Hiligaynon, and Ilocano
merged_df_cleaned= merged_df_cleaned[
    (merged_df_cleaned['language'].isin(language_codes))
    ]

In [193]:
# # Show rows with language not in the specified language codes
# merged_df_cleaned = merged_df_cleaned[~merged_df_cleaned['language'].isin(language_codes)]
# merged_df_cleaned

In [194]:
base_df = merged_df_cleaned.reset_index(drop=True).copy()
n_rows = len(base_df)
ingested_at = pd.Timestamp.now(tz="UTC")

# Keep source created_at when available, otherwise fall back to date_posted then ingestion time
source_created_at = pd.to_datetime(base_df.get("created_at"), errors="coerce", utc=True)
effective_created_at = source_created_at.fillna(base_df["date_posted"])
effective_created_at = effective_created_at.fillna(ingested_at)

# Metrics are already mapped from raw source-specific columns in earlier cells
view_count = to_nullable_int(base_df.get("view_count", pd.Series([pd.NA] * n_rows)))
like_count = to_nullable_int(base_df.get("like_count", pd.Series([pd.NA] * n_rows)))
share_count = to_nullable_int(base_df.get("share_count", pd.Series([pd.NA] * n_rows)))
comment_count = to_nullable_int(base_df.get("comment_count", pd.Series([pd.NA] * n_rows)))

posts_df = pd.DataFrame({
    "id": [str(uuid.uuid4()) for _ in range(n_rows)],
    "source": base_df["source"].astype(str).str.lower(),
    "external_post_id": base_df["external_post_id"].astype(str),
    "text": base_df["text"],
    "cleaned_text": base_df.get("cleaned_text", base_df["text"]),
    "language": base_df.get("language", pd.Series([pd.NA] * n_rows)),
    "date_posted": base_df["date_posted"],
    "date_collected": [ingested_at] * n_rows,
    "view_count": view_count,
    "like_count": like_count,
    "share_count": share_count,
    "comment_count": comment_count,
    "created_at": effective_created_at,
    "updated_at": [ingested_at] * n_rows,
})

merged_df_final = posts_df[POSTS_COLUMNS]


In [195]:
merged_df_final

,id,source,external_post_id,text,cleaned_text,language,date_posted,date_collected,view_count,like_count,share_count,comment_count,created_at,updated_at
0,474ae30f-ec01-492d-ab36-62be25540a3f,twitter,1969912891831939529,anong gusto mo gawin niya makipagbarda sa mga ...,anong gusto mo gawin niya makipagbarda sa mga ...,tl,2025-09-21 23:53:27+00:00,2026-06-29 02:47:47.291693+00:00,<NA>,<NA>,<NA>,<NA>,2025-09-21 23:53:27+00:00,2026-06-29 02:47:47.291693+00:00
1,61c6bbd8-c692-4e85-8d16-378fd71b8489,twitter,1969904337368256777,@MaseDenver @MaseDenver thoughts on the offici...,thoughts on the officiating crew tonight? that...,en,2025-09-21 23:19:27+00:00,2026-06-29 02:47:47.291693+00:00,<NA>,<NA>,<NA>,<NA>,2025-09-21 23:19:27+00:00,2026-06-29 02:47:47.291693+00:00
2,f4dbfa06-e804-4ca7-89f3-b87d2f56b124,twitter,1969903982794743937,@VicLombardi @Broncos @AltitudeTV Horrible gam...,horrible game when it comes to the vibe and a ...,en,2025-09-21 23:18:03+00:00,2026-06-29 02:47:47.291693+00:00,<NA>,<NA>,<NA>,<NA>,2025-09-21 23:18:03+00:00,2026-06-29 02:47:47.291693+00:00
3,1b33a059-8a41-4b07-9a7f-9b5615fe00f6,twitter,1969902984105787481,@ZacStevensDNVR Shit game. There’s no other wa...,shit game. theres no other way to put it. refs...,en,2025-09-21 23:14:05+00:00,2026-06-29 02:47:47.291693+00:00,<NA>,<NA>,<NA>,<NA>,2025-09-21 23:14:05+00:00,2026-06-29 02:47:47.291693+00:00
4,ceb3b223-fb6b-4b02-98dd-ddae895aedc4,twitter,1969902370755944783,@MileHighReport Nix’s accuracy on deep throws ...,nixs accuracy on deep throws was ass.,en,2025-09-21 23:11:38+00:00,2026-06-29 02:47:47.291693+00:00,<NA>,<NA>,<NA>,<NA>,2025-09-21 23:11:38+00:00,2026-06-29 02:47:47.291693+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40477,9193c8c1-9572-423c-b51d-45bda4cc000d,tiktok,7615164715752246548,sapy kuma medeknam ngu eh teg-in😔 #fypシ #copy...,sapy kuma medeknam ngu eh teg-in,en,2026-03-09 08:08:33+00:00,2026-06-29 02:47:47.291693+00:00,393,<NA>,1,2,2026-03-09 08:08:33+00:00,2026-06-29 02:47:47.291693+00:00
40478,9bce0b4d-80bf-4597-b2bf-121e9bc8ac43,tiktok,7615149486167330069,Parang ice tubig sa lamig 😆❄️ pero yung view i...,parang ice tubig sa lamig pero yung view ibang...,tl,2026-03-09 07:09:26+00:00,2026-06-29 02:47:47.291693+00:00,2926,<NA>,2,6,2026-03-09 07:09:26+00:00,2026-06-29 02:47:47.291693+00:00
40479,35cdda06-bff8-4956-8aaa-8fb0e2646b21,tiktok,7614809016413211924,#duet with @Cortez motovlog😭😭😭😭😭😭😭😭😭😭😭😭😭,with motovlog,en,2026-03-08 09:08:20+00:00,2026-06-29 02:47:47.291693+00:00,4885,<NA>,16,8,2026-03-08 09:08:20+00:00,2026-06-29 02:47:47.291693+00:00
40480,9ce87219-b0f6-4bf4-9b41-e2e74ae00659,tiktok,7614770168417291541,#CapCut #fyp mga mananabtan 🙏🏼 #fypppppppppppp...,mga mananabtan,ceb,2026-03-08 06:37:29+00:00,2026-06-29 02:47:47.291693+00:00,387,<NA>,3,4,2026-03-08 06:37:29+00:00,2026-06-29 02:47:47.291693+00:00


In [196]:
merged_df_final.to_csv('../data/processed/merged_data.csv', index=False)

In [197]:
merged_df_final

,id,source,external_post_id,text,cleaned_text,language,date_posted,date_collected,view_count,like_count,share_count,comment_count,created_at,updated_at
0,474ae30f-ec01-492d-ab36-62be25540a3f,twitter,1969912891831939529,anong gusto mo gawin niya makipagbarda sa mga ...,anong gusto mo gawin niya makipagbarda sa mga ...,tl,2025-09-21 23:53:27+00:00,2026-06-29 02:47:47.291693+00:00,<NA>,<NA>,<NA>,<NA>,2025-09-21 23:53:27+00:00,2026-06-29 02:47:47.291693+00:00
1,61c6bbd8-c692-4e85-8d16-378fd71b8489,twitter,1969904337368256777,@MaseDenver @MaseDenver thoughts on the offici...,thoughts on the officiating crew tonight? that...,en,2025-09-21 23:19:27+00:00,2026-06-29 02:47:47.291693+00:00,<NA>,<NA>,<NA>,<NA>,2025-09-21 23:19:27+00:00,2026-06-29 02:47:47.291693+00:00
2,f4dbfa06-e804-4ca7-89f3-b87d2f56b124,twitter,1969903982794743937,@VicLombardi @Broncos @AltitudeTV Horrible gam...,horrible game when it comes to the vibe and a ...,en,2025-09-21 23:18:03+00:00,2026-06-29 02:47:47.291693+00:00,<NA>,<NA>,<NA>,<NA>,2025-09-21 23:18:03+00:00,2026-06-29 02:47:47.291693+00:00
3,1b33a059-8a41-4b07-9a7f-9b5615fe00f6,twitter,1969902984105787481,@ZacStevensDNVR Shit game. There’s no other wa...,shit game. theres no other way to put it. refs...,en,2025-09-21 23:14:05+00:00,2026-06-29 02:47:47.291693+00:00,<NA>,<NA>,<NA>,<NA>,2025-09-21 23:14:05+00:00,2026-06-29 02:47:47.291693+00:00
4,ceb3b223-fb6b-4b02-98dd-ddae895aedc4,twitter,1969902370755944783,@MileHighReport Nix’s accuracy on deep throws ...,nixs accuracy on deep throws was ass.,en,2025-09-21 23:11:38+00:00,2026-06-29 02:47:47.291693+00:00,<NA>,<NA>,<NA>,<NA>,2025-09-21 23:11:38+00:00,2026-06-29 02:47:47.291693+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40477,9193c8c1-9572-423c-b51d-45bda4cc000d,tiktok,7615164715752246548,sapy kuma medeknam ngu eh teg-in😔 #fypシ #copy...,sapy kuma medeknam ngu eh teg-in,en,2026-03-09 08:08:33+00:00,2026-06-29 02:47:47.291693+00:00,393,<NA>,1,2,2026-03-09 08:08:33+00:00,2026-06-29 02:47:47.291693+00:00
40478,9bce0b4d-80bf-4597-b2bf-121e9bc8ac43,tiktok,7615149486167330069,Parang ice tubig sa lamig 😆❄️ pero yung view i...,parang ice tubig sa lamig pero yung view ibang...,tl,2026-03-09 07:09:26+00:00,2026-06-29 02:47:47.291693+00:00,2926,<NA>,2,6,2026-03-09 07:09:26+00:00,2026-06-29 02:47:47.291693+00:00
40479,35cdda06-bff8-4956-8aaa-8fb0e2646b21,tiktok,7614809016413211924,#duet with @Cortez motovlog😭😭😭😭😭😭😭😭😭😭😭😭😭,with motovlog,en,2026-03-08 09:08:20+00:00,2026-06-29 02:47:47.291693+00:00,4885,<NA>,16,8,2026-03-08 09:08:20+00:00,2026-06-29 02:47:47.291693+00:00
40480,9ce87219-b0f6-4bf4-9b41-e2e74ae00659,tiktok,7614770168417291541,#CapCut #fyp mga mananabtan 🙏🏼 #fypppppppppppp...,mga mananabtan,ceb,2026-03-08 06:37:29+00:00,2026-06-29 02:47:47.291693+00:00,387,<NA>,3,4,2026-03-08 06:37:29+00:00,2026-06-29 02:47:47.291693+00:00
